# Conditional-VAE-Augmented Classifier

Three independent outputs are trained with BCE-based loss. Validation chooses the checkpoint; test is used only at the end.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report

sys.path.append(str(Path.cwd()))
from multilabel_utils import (
    CLASS_NAMES,
    LABEL_COLUMNS,
    MultilabelDataset,
    calculate_multilabel_metrics,
    classifier_transform,
    create_resnet18,
    get_device,
    predict_multilabel,
    set_seed,
    train_classifier,
)

SEED = 42
THRESHOLD = 0.5
EPOCHS = 5
set_seed(SEED)
device = get_device()
PROJECT_ROOT = Path.cwd().parents[1]
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "multilabel"
MODEL_DIR = PROJECT_ROOT / "models" / "multilabel"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")
print("Device:", device)
print("Split sizes:", len(train_df), len(val_df), len(test_df))

Device: mps
Split sizes: 1503 302 300


In [2]:
synthetic_df = pd.read_csv(PROJECT_ROOT / "data" / "synthetic" / "multilabel_vae" / "metadata.csv")
train_df["is_synthetic"] = False
train_df["image_path"] = ""
augmented_train_df = pd.concat([train_df, synthetic_df], ignore_index=True, sort=False)
print("Real training images:", len(train_df))
print("Synthetic training images:", len(synthetic_df))
print(augmented_train_df[LABEL_COLUMNS].sum())

transform = classifier_transform()
train_loader = DataLoader(
    MultilabelDataset(augmented_train_df, PROJECT_ROOT, transform),
    batch_size=32,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)
val_loader = DataLoader(MultilabelDataset(val_df, PROJECT_ROOT, transform), batch_size=32)
test_loader = DataLoader(MultilabelDataset(test_df, PROJECT_ROOT, transform), batch_size=32)

Real training images: 1503
Synthetic training images: 500
Surface_Crack    1665
Delamination      446
Pinhole           658
dtype: int64


In [3]:
model = create_resnet18(len(LABEL_COLUMNS)).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
checkpoint_path = MODEL_DIR / "vae_augmented_best.pth"

history = train_classifier(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    checkpoint_path,
    epochs=EPOCHS,
)

Epoch 1/5 | Train loss: 0.3548 | Validation loss: 0.2121


Epoch 2/5 | Train loss: 0.2034 | Validation loss: 0.1582


Epoch 3/5 | Train loss: 0.1921 | Validation loss: 0.1571


Epoch 4/5 | Train loss: 0.1600 | Validation loss: 0.1369


Epoch 5/5 | Train loss: 0.1570 | Validation loss: 0.2244


In [4]:
model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
true_labels, probabilities, predictions = predict_multilabel(
    model, test_loader, device, threshold=THRESHOLD
)

print(classification_report(
    true_labels,
    predictions,
    target_names=CLASS_NAMES,
    zero_division=0,
))

metrics = calculate_multilabel_metrics(true_labels, predictions, "VAE Augmentation")
metrics_df = pd.DataFrame([metrics])
metrics_df.to_csv(PROCESSED_DIR / "vae_augmented_metrics.csv", index=False)
metrics_df.round(4)

               precision    recall  f1-score   support

Surface Crack       1.00      0.96      0.98       276
 Delamination       1.00      0.58      0.73        26
      Pinhole       0.96      0.92      0.94        73

    micro avg       0.99      0.92      0.95       375
    macro avg       0.98      0.82      0.88       375
 weighted avg       0.99      0.92      0.95       375
  samples avg       0.99      0.95      0.96       375



,Model,Exact Match Accuracy,Hamming Loss,Micro F1,Macro F1,Surface Crack F1,Delamination F1,Pinhole F1
0,VAE Augmentation,0.9033,0.0367,0.9545,0.8816,0.976,0.7317,0.9371


## Interpretation

Focus on macro F1 and the minority-label F1 scores. Exact-match accuracy requires the entire three-value vector to be correct.